# REPA × CheXpert × MedDINOv3

Trains **SiT-S/8** on CheXpert chest X-rays using **MedDINOv3 ViT-Base** as the representation encoder.

**Requirements (Kaggle datasets attached to this notebook):**
- `nikiboura/meddinov3` — contains MedDINOv3 code + checkpoint
- `ashery/chexpert` — CheXpert-v1.0-small chest X-ray dataset

All code runs from the cloned `nikiboura/REPA` fork.

FID output = 544

## 1. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops pandas

## 2. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
#check if folder exists already
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
#latest git commit
echo "REPA ready. Commit: "

## 3. Set paths

In [ ]:
import os

#point to REPA, CheXpert, MedDINOv3
REPA_DIR      = '/kaggle/working/REPA'
DATA_DIR      = '/kaggle/working/data/chexpert_256'
CHEXPERT_ROOT = '/kaggle/input/datasets/ashery/chexpert'
MEDDINOV3     = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
CKPT_PATH     = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

#create environmental variables
os.environ['MEDDINOV3_PATH'] = MEDDINOV3
os.environ['MEDDINOV3_CKPT'] = CKPT_PATH

#create directory for dataset
os.makedirs(DATA_DIR, exist_ok=True)

#sanity checks
print(f'MedDINOv3 exists  : {os.path.isdir(MEDDINOV3)}')
print(f'Checkpoint exists : {os.path.isfile(CKPT_PATH)}')
print(f'CheXpert exists   : {os.path.isdir(CHEXPERT_ROOT)}')
print(f'train.csv exists  : {os.path.isfile(os.path.join(CHEXPERT_ROOT, "train.csv"))}')

## 4. Prepare CheXpert
Reads chest X-ray images, converts to RGB, resizes to 256×256, VAE-encodes, saves latents as .

Binary labels: **0 = normal** ("No Finding"), **1 = abnormal** (any pathology).

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '/kaggle/working/REPA/prepare_chexpert.py',
    '--out-dir',       '/kaggle/working/data/chexpert_256',
    '--chexpert-root', '/kaggle/input/datasets/ashery/chexpert',
    '--resolution',   '256',
    '--max-samples',  '5000',
]
result = subprocess.run(cmd, text=True)
print('Exit code:', result.returncode)

## 5. Train
Runs 15000 steps. Checkpoints saved to .

**Models**
1. SiT-S/8 (Diffusion Model): denoises in latent space
2. MedDINOv3 ViT-Base: pretrained medical encoder

**Losses**:
1. denoising_loss: standard diffusion loss
2. proj_loss: REPA alignment loss

Total loss = denoising_loss + proj_loss * proj_coeff

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()
env['MEDDINOV3_PATH'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
env['MEDDINOV3_CKPT'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/kaggle/working/REPA/train.py',
    '--exp-name',            'repa-sits8-meddinov3-chexpert',
    '--output-dir',          '/kaggle/working/results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '2',
    '--encoder-depth',       '8',
    '--enc-type',            'meddinov3-vit-b',
    '--data-dir',            '/kaggle/working/data/chexpert_256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '15000',
    '--checkpointing-steps', '15000',
    '--sampling-steps',      '99999999',
]

process = subprocess.Popen(cmd, cwd='/kaggle/working/REPA', env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

last_step = 0
pbar = tqdm(total=15000, desc='Training')
for line in process.stdout:
    m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
pbar.close()
process.wait()
print('Training complete. Checkpoint saved.')

## 6. Check checkpoint

In [ ]:
import glob
print(glob.glob('/kaggle/working/results/**/*.pt', recursive=True))

## 7. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()
env['MEDDINOV3_PATH'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal'
env['MEDDINOV3_CKPT'] = '/kaggle/input/datasets/nikiboura/meddinov3-minimal/meddinov3_minimal/checkpoints/model.pth'

ckpts = sorted(glob.glob('/kaggle/working/results/repa-sits8-meddinov3-chexpert/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/kaggle/working/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '2',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
]

result = subprocess.run(cmd, cwd='/kaggle/working/REPA', env=env, text=True)
print('Exit code:', result.returncode)

## 8. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))

#loads images
data = np.load(npz_files[-1])
imgs = data[data.files[0]]

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 9. Check FID Score

In [ ]:
import subprocess, glob, numpy as np, os
from PIL import Image
from tqdm import tqdm

subprocess.run(['pip', 'install', '-q', 'torch-fidelity'], check=True)

gen_npz = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated:', gen_npz)

#save generated images to folder
gen_dir = '/kaggle/working/gen_images'
os.makedirs(gen_dir, exist_ok=True)
data = np.load(gen_npz)
imgs = data[data.files[0]]
for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
    Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

real_dir = '/kaggle/working/data/chexpert_256/images/train'

subprocess.run([
    'fidelity',
    '--gpu', '0',
    '--fid',
    '--input1', gen_dir,
    '--input2', real_dir,
], text=True)